In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetV2B2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
print("TensorFlow Version:", tf.__version__)
try:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        strategy = tf.distribute.MirroredStrategy()
        print(f"GPUs detected: {len(gpus)}. Using MirroredStrategy.")
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    else:
        strategy = tf.distribute.get_strategy()
        print("Warning: No GPU detected. Training will be run on CPU and will be very slow.")
except Exception as e:
    print("Could not initialize GPU strategy, falling back to default strategy.", e)
    strategy = tf.distribute.get_strategy()
gc.enable()
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

TensorFlow Version: 2.19.0
GPUs detected: 1. Using MirroredStrategy.
Could not initialize GPU strategy, falling back to default strategy. Physical devices cannot be modified after being initialized


In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"tirthendu21","key":"6deaaf003f2ae9a6d3ba13ad34b81256"}'}

In [ ]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets list

ref                                                         title                                                  size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------------------------  -----------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
dmahajanbe23/bmw-global-automotive-sales                    BMW Global Automotive Sales                           55017  2026-02-22 18:18:38.170000           3358         62  1.0              
shree0910/online-vs-in-store-shopping-behaviour-dataset     Online vs In-Store Shopping Behaviour Dataset        354896  2026-02-18 08:16:20.137000           1792         42  1.0              
krupalpatel07/gold-price-dynamics                           Gold Price Dynamics                                   85982  2026-03-03 05:42:46.960000            622         24  1.0              
likithagedipudi/starbucks-customer-

In [ ]:
!kaggle datasets download -d orvile/ultrasound-fetus-dataset

Dataset URL: https://www.kaggle.com/datasets/orvile/ultrasound-fetus-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
ultrasound-fetus-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
import zipfile

with zipfile.ZipFile("ultrasound-fetus-dataset.zip", "r") as zip_ref:
    zip_ref.extractall("dataset")

In [ ]:
import os
print(os.listdir("dataset/Ultrasound Fetus Dataset"))

['Ultrasound Fetus Dataset']


In [ ]:
import os
print(os.listdir("dataset"))

['Ultrasound Fetus Dataset', 'ultrasound_fetus.csv']


In [ ]:
import os
print(os.listdir("dataset/Ultrasound Fetus Dataset"))

['Ultrasound Fetus Dataset']


In [ ]:
print(os.listdir("dataset/Ultrasound Fetus Dataset/Ultrasound Fetus Dataset"))

['Data']


In [ ]:
print(os.listdir("dataset/Ultrasound Fetus Dataset/Ultrasound Fetus Dataset/Data"))

['Data']


In [ ]:
print(os.listdir("dataset/Ultrasound Fetus Dataset/Ultrasound Fetus Dataset/Data/Data"))

['Resnet_fineTuning.pth', 'FetusDataset.csv', 'OverlayedImages', 'train', 'validation', 'test', 'Datasets']


In [ ]:
train_dir = "dataset/Ultrasound Fetus Dataset/Ultrasound Fetus Dataset/Data/Data/train"

print(os.listdir(train_dir))

['normal', 'benign', 'malignant']


In [ ]:
import os
root_dir = "dataset/Ultrasound Fetus Dataset/Ultrasound Fetus Dataset/Data/Data"
image_ext = (".png", ".jpg", ".jpeg")
for folder in os.listdir(root_dir):
    folder_path = os.path.join(root_dir, folder)
    if os.path.isdir(folder_path):
        print(f"\n{folder.upper()}")
        for sub in os.listdir(folder_path):
            sub_path = os.path.join(folder_path, sub)
            if os.path.isdir(sub_path):
                files = [f for f in os.listdir(sub_path) if f.lower().endswith(image_ext)]
                print(f"{sub} : {len(files)}")


OVERLAYEDIMAGES
normal : 50
benign : 51
malignant : 300

TRAIN
normal : 242
benign : 241
malignant : 1443

VALIDATION
normal : 43
benign : 43
malignant : 255

TEST
normal : 50
benign : 51
malignant : 300

DATASETS
normal : 365
benign : 390
malignant : 2209


In [ ]:
import tensorflow as tf

TRAIN_DIR = "dataset/Ultrasound Fetus Dataset/Ultrasound Fetus Dataset/Data/Data/train"
VAL_DIR   = "dataset/Ultrasound Fetus Dataset/Ultrasound Fetus Dataset/Data/Data/validation"
TEST_DIR  = "dataset/Ultrasound Fetus Dataset/Ultrasound Fetus Dataset/Data/Data/test"

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

Found 1926 files belonging to 3 classes.
Found 341 files belonging to 3 classes.
Found 401 files belonging to 3 classes.


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Sequential
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])
train_ds = train_ds.map(lambda x, y: (data_aug(x), y))

In [ ]:
train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y))
val_ds   = val_ds.map(lambda x, y: (preprocess_input(x), y))
test_ds  = test_ds.map(lambda x, y: (preprocess_input(x), y))

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
labels = []
for _, y in train_ds:
    labels.extend(np.argmax(y.numpy(), axis=1))
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
scale_factor = 0.6 
class_weight_dict = {k: v*scale_factor for k, v in enumerate(class_weights)}
print("Adjusted class weights:", class_weight_dict)

Adjusted class weights: {0: np.float64(1.5983402489626555), 1: np.float64(0.26694386694386696), 2: np.float64(1.5917355371900825)}


In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization

In [ ]:
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

In [ ]:
base_model.trainable = False

In [ ]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

In [ ]:
model.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,427,459 (9.26 MB)

 Trainable params: 166,915 (652.01 KB)

 Non-trainable params: 2,260,544 (8.62 MB)

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
class_weight = {
    0: 2.7,
    1: 0.45,
    2: 2.7
}

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weight_dict  
)

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight_dict   
)

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)

print("Test Accuracy:", test_acc)

13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 400ms/step - accuracy: 0.6468 - loss: 5.2208
Test Accuracy: 0.748129665851593


In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(["Train","Validation"])
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np
y_true = []
y_pred = []
for x, y in test_ds:
    preds = model.predict(x)
    y_true.extend(np.argmax(y.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))
cm = confusion_matrix(y_true, y_pred)
print(cm)